# 🔺 VÉRTICE • Laboratório de superfície de ataque
**Cada ativo, uma origem. Cada relação, uma evidência.**

Execute **Ambiente de execução → Executar tudo**. A demonstração usa dados sintéticos e não consulta alvos. Não exige GPU, API paga, ngrok ou instalação no seu computador.

Você verá um dossiê navegável, duas coletas simuladas comparáveis e um pacote para download. A coleta real é opcional e vem desativada. Os arquivos do runtime são temporários; baixe o pacote antes de encerrar.


In [ ]:
#@title 1. Preparar o VÉRTICE
import os, sys, subprocess, json, tempfile
from pathlib import Path
BASE = Path('/content') if Path('/content').exists() else Path(tempfile.mkdtemp(prefix='vertice-colab-'))
REPO = BASE / 'VeRTICE'
URL = 'https://github.com/Ridd1kulusC0d3r/VeRTICE.git'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import vertice
print('VÉRTICE', vertice.__version__)
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
print('Reexecuções reutilizam este checkout. Para atualizar, reinicie o runtime.')


In [ ]:
#@title 2. Criar demonstração offline e verificar integridade
from vertice.demo_data import build_demo
from vertice.core.workspace import Workspace
from vertice.report import render_report
WORKSPACE = BASE / 'vertice-demo'
if not (WORKSPACE / 'meta.json').exists():
    build_demo(str(WORKSPACE)).close()
ws = Workspace(str(WORKSPACE))
try:
    print(json.dumps(ws.evidence.verify_chain(), indent=2))
    (WORKSPACE / 'dossie.html').write_text(render_report(ws), encoding='utf-8')
finally:
    ws.close()


## O que mudou entre duas coletas?
Simulamos um serviço antigo que deixou de aparecer e um novo subdomínio. Isso demonstra **observações**, sem concluir que o serviço antigo foi removido. A simulação não faz rede.


In [ ]:
#@title 3. Comparar duas execuções simuladas
from vertice.core.engine import Engine
from vertice.core.history import compare
from vertice.collectors.base import Collector, Finding

class SimulatedSource(Collector):
    name = 'demo-simulation'
    etypes = ('domain',)
    def __init__(self, hosts):
        self.hosts = hosts
    def collect(self, entity):
        return [Finding(h, 'host', 'has-subdomain', raw={'synthetic': True}) for h in self.hosts]

ws = Workspace(str(WORKSPACE))
try:
    ids = []
    for hosts in [('api.aurora.example', 'old.aurora.example'),
                  ('api.aurora.example', 'staging.aurora.example')]:
        engine = Engine(ws, selected=['pivot'])
        engine.collectors = [SimulatedSource(hosts)]
        ids.append(engine.run(['aurora.example'])['run_id'])
    difference = compare(ws, *ids)
    print(json.dumps(difference, indent=2, ensure_ascii=False))
    (WORKSPACE / 'comparison.json').write_text(json.dumps(difference, indent=2, ensure_ascii=False), encoding='utf-8')
    (WORKSPACE / 'dossie.html').write_text(render_report(ws), encoding='utf-8')
finally:
    ws.close()


In [ ]:
#@title 4. Abrir bancada web dentro do Colab
import threading
from http.server import HTTPServer
from vertice.bench import _handler
# Reexecutar esta célula encerra a instância anterior.
if 'server' in globals():
    server.shutdown()
    server.server_close()
    bench_ws.close(save=False)
bench_ws = Workspace(str(WORKSPACE))
server = HTTPServer(('127.0.0.1', 0), _handler(bench_ws))
threading.Thread(target=server.serve_forever, daemon=True).start()
PORT = server.server_port
from urllib.request import urlopen
with urlopen(f'http://127.0.0.1:{PORT}/verify', timeout=10) as response:
    assert json.load(response)['ok']
try:
    from google.colab import output
except ImportError:
    print(f'Bancada local: http://127.0.0.1:{PORT}')
else:
    output.serve_kernel_port_as_iframe(PORT, height=850)
print('Se o iframe não aparecer, baixe o ZIP abaixo e abra dossie.html.')


In [ ]:
#@title 5. Exportar relatório, grafo e evidências (sem credenciais)
baixar_agora = False #@param {type:'boolean'}
import zipfile
ws = Workspace(str(WORKSPACE))
try:
    (WORKSPACE / 'dossie.html').write_text(render_report(ws), encoding='utf-8')
    (WORKSPACE / 'integrity.json').write_text(json.dumps(ws.evidence.verify_chain(), indent=2), encoding='utf-8')
finally:
    ws.close()
archive = BASE / 'vertice-demo.zip'
allowed = ['dossie.html', 'graph.json', 'evidence.db', 'meta.json', 'comparison.json', 'integrity.json']
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for name in allowed:
        if (WORKSPACE / name).exists():
            bundle.write(WORKSPACE / name, name)
print('Pacote pronto:', archive)
print('No Colab, abra Arquivos na barra lateral para baixar, ou marque baixar_agora e execute esta célula.')
if baixar_agora:
    from google.colab import files
    files.download(str(archive))


## Coleta real opcional
Preencha um domínio autorizado e marque a opção. Esta célula consulta apenas crt.sh, DNS via resolvedor e RDAP, sem fingerprint direto. DNS via resolvedor pode resultar em consulta ao servidor autoritativo: não significa invisibilidade. Serviços externos podem impor bloqueios, limites ou falhar.

O workspace real é separado da demo. Para visualizar/exportar o caso real, execute `WORKSPACE = REAL_WORKSPACE` em uma nova célula e reexecute as células 4 e 5. Não execute a simulação (célula 3) nesse workspace.


In [ ]:
#@title 6. Coleta real (desativada por padrão)
executar_coleta_real = False #@param {type:'boolean'}
dominio = '' #@param {type:'string'}
autorizacao = '' #@param {type:'string'}
if executar_coleta_real:
    from vertice.core.entities import make
    seed = make(dominio)
    if seed is None or seed.etype != 'domain' or not autorizacao.strip():
        raise ValueError('Informe um domínio válido e a referência de autorização.')
    REAL_WORKSPACE = BASE / ('vertice-real-' + seed.value)
    real = Workspace.create(str(REAL_WORKSPACE), name=seed.value,
                            scope=[seed.value], authorized_by=autorizacao)
    try:
        result = Engine(real, selected=['crtsh', 'dns', 'rdap'],
                        max_workers=3, max_tasks=3, log=print).run([seed.value], depth=1)
        print(json.dumps(result, indent=2))
        (REAL_WORKSPACE / 'dossie.html').write_text(render_report(real), encoding='utf-8')
    finally:
        real.close()
else:
    print('Coleta real desativada. A demonstração acima é totalmente sintética.')


## Próximos testes
A CLI aceita `collectors`, `runs`, `diff`, `import --format subfinder` e `import --format nmap`. Consulte o README para comandos completos. Importação não executa ferramentas nem autoriza os ativos importados.

A verificação de hashes cobre integridade interna; preserve `integrity.json` em outro local para comparação externa. Não é assinatura digital nem certificação forense.
